# Week 66, Day 4: Race conditions and locks

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w066d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from itertools import combinations

def interleavings(a, b):
    n = len(a) + len(b)
    for positions in combinations(range(n), len(a)):
        merged, ia, ib = [], iter(a), iter(b)
        for i in range(n):
            merged.append(next(ia) if i in positions else next(ib))
        yield merged

def run(order):
    x, reg = 0, {}
    for who, op in order:
        if op == "load":
            reg[who] = x
        else:
            x = reg[who] + 1
    return x

A = [("A", "load"), ("A", "store")]
B = [("B", "load"), ("B", "store")]
results = [run(order) for order in interleavings(A, B)]
print(len(results), sorted(set(results)), results.count(1))

In [ ]:
from math import comb

for m, n in [(2, 2), (3, 3), (10, 10)]:
    print(f"{m} and {n} steps: {comb(m + n, m)} interleavings")

balance = 100

def withdraw_steps(amount):
    "Check, then act: two steps that must not be separated."
    return [("check", amount), ("take", amount)]

def run(order):
    bal, approved = 100, {}
    for who, (step, amount) in order:
        if step == "check":
            approved[who] = bal >= amount
        elif approved[who]:
            bal -= amount
    return bal

a = [("A", s) for s in withdraw_steps(80)]
b = [("B", s) for s in withdraw_steps(80)]
print(run(a + b), run([a[0], b[0], a[1], b[1]]))     # in turn, then interleaved: overdrawn

In [ ]:
import threading

lock = threading.Lock()
with lock:
    print("held:", lock.locked(), "| a second acquire succeeds:", lock.acquire(blocking=False))
print("after with:", lock.locked())

rlock = threading.RLock()

class Account:
    def __init__(self):
        self.balance, self.lock = 100, threading.RLock()
    def withdraw(self, amount):
        with self.lock:
            if self.balance >= amount:
                self.balance -= amount
                return True
            return False
    def withdraw_twice(self, amount):
        with self.lock:                       # the same thread takes the lock again inside withdraw
            return self.withdraw(amount) and self.withdraw(amount)

acc = Account()
print(acc.withdraw_twice(30), acc.balance)

In [ ]:
cache, loads = {}, []

def load(key):
    loads.append(key)
    return key.upper()

def get_racy(steps, key):
    "Each thread checks, then (after a possible switch) stores."
    seen_missing = {}
    for who in steps:
        if who not in seen_missing:
            seen_missing[who] = key not in cache       # check
        elif seen_missing[who]:
            cache[key] = load(key)                     # act
            seen_missing[who] = False
    return len(loads)

print(get_racy(["A", "B", "A", "B"], "tea"), "loads instead of 1")

cache.clear(); loads.clear()
def get_atomic(key):
    if key not in cache:                               # imagine this whole function inside one lock
        cache[key] = load(key)
    return cache[key]
get_atomic("tea"); get_atomic("tea")
print(len(loads))

In [ ]:
def wait_for_graph(holds, wants):
    "holds: lock -> thread holding it; wants: thread -> lock it's waiting for."
    return {t: holds[lock] for t, lock in wants.items() if lock in holds}

def find_cycle(graph):
    for start in graph:
        seen, node = [], start
        while node in graph and node not in seen:
            seen.append(node)
            node = graph[node]
        if node in seen:
            return seen[seen.index(node):]
    return []

holds = {"A": "t1", "B": "t2"}
wants = {"t1": "B", "t2": "A"}
print(find_cycle(wait_for_graph(holds, wants)))

def lock_order(a, b):
    return tuple(sorted((a, b)))                 # the same order whichever way money moves
print(lock_order("x", "y"), lock_order("y", "x"))

In [ ]:
import threading

slots = threading.Semaphore(2)
taken = [slots.acquire(blocking=False) for _ in range(3)]
print(taken)
slots.release(); slots.release()

bounded = threading.BoundedSemaphore(1)
bounded.acquire()
bounded.release()
try:
    bounded.release()
except ValueError as e:
    print("ValueError:", e)

cond = threading.Condition()
with cond:
    print("waiting returns:", cond.wait(timeout=0))   # nobody will notify: times out at once

> **Key idea.** A race condition is a bug whose result depends on how threads interleave; read-modify-write (+=) and check-then-act are the usual forms, and enumerating interleavings of small cases shows them. A critical section must run in one thread at a time: protect it with a Lock, always with with, around every access to the shared data, kept short; acquire(blocking=False) and timeout avoid waiting for ever; RLock allows the owning thread to re-acquire. Deadlock is a cycle of threads waiting for each other's locks: prevent it with one global lock order (sorted), fewer locks, timeouts and no callbacks under locks, and detect it as a cycle in the wait-for graph. Semaphore limits concurrency, Condition waits for a state, Barrier synchronises a group; queues and immutable data avoid sharing altogether.

### Exercise: Every possible order

Write `interleavings(a, b)`: a list of every way to merge the step lists `a` and `b` that keeps each list's own order (as lists), in the order produced by choosing, at each point, a step from `a` before a step from `b`. Then `count_interleavings(m, n)`: how many there are, computed with `math.comb` (not by generating them).

In [ ]:
import math

def interleavings(a, b):
    ___

def count_interleavings(m, n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_interleave():\n    \"enumerating interleavings\"\n    assert interleavings([1, 2], [\"x\"]) == [[1, 2, \"x\"], [1, \"x\", 2], [\"x\", 1, 2]]\n    assert len(interleavings([1, 2, 3], [4, 5, 6])) == 20 == count_interleavings(3, 3)\n    assert interleavings([], [1]) == [[1]] and interleavings([], []) == [[]]\n    assert count_interleavings(10, 10) == 184756\n    for merged in interleavings([\"a1\", \"a2\"], [\"b1\", \"b2\"]):\n        assert merged.index(\"a1\") < merged.index(\"a2\") and merged.index(\"b1\") < merged.index(\"b2\")")

### Exercise: All possible results

Model threads as lists of instructions on shared variables, each thread with its own register: `("load", var)` copies a variable into the register, `("add", k)` adds `k` to the register, and `("store", var)` writes the register to the variable. Write `execute(order, shared)`: run a list of `(thread, instruction)` pairs on a copy of the dict `shared` and return the final dict. Then `possible(threads, shared, var)`: the sorted list of distinct final values of `var` over **all** interleavings of the threads (a dict from thread name to instruction list; any number of threads).

In [ ]:
def execute(order, shared):
    ___

def possible(threads, shared, var):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_outcomes():\n    \"finding races\"\n    inc = [(\"load\", \"x\"), (\"add\", 1), (\"store\", \"x\")]\n    assert possible({\"A\": inc, \"B\": inc}, {\"x\": 0}, \"x\") == [1, 2]\n    assert possible({\"A\": inc, \"B\": inc, \"C\": inc}, {\"x\": 0}, \"x\") == [1, 2, 3]\n    dec = [(\"load\", \"x\"), (\"add\", -5), (\"store\", \"x\")]\n    assert possible({\"A\": inc, \"B\": dec}, {\"x\": 10}, \"x\") == [5, 6, 11]\n    assert execute([(\"A\", (\"load\", \"x\")), (\"A\", (\"add\", 2)), (\"A\", (\"store\", \"y\"))], {\"x\": 1}) == {\"x\": 1, \"y\": 3}\n    start = {\"x\": 0}\n    execute([(\"A\", (\"load\", \"x\")), (\"A\", (\"store\", \"x\"))], start)\n    assert start == {\"x\": 0} and possible({\"A\": inc}, {\"x\": 7}, \"x\") == [8]")

### Exercise: Locks in the model

Extend the model with locks: `("acquire", name)` can only run when no **other** thread holds that lock (then this thread holds it), and `("release", name)` gives it up. Write `possible_locked(threads, shared, var)`: like `possible`, but only exploring orders the locks allow (an interleaving step is available only if it isn't an `acquire` of a lock held by another thread). Then `protect(steps, lock="L")`: the steps wrapped in an acquire and a release. With protection, the counter race disappears.

In [ ]:
def possible_locked(threads, shared, var):
    ___

def protect(steps, lock="L"):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_locked():\n    \"locks remove the race\"\n    inc = [(\"load\", \"x\"), (\"add\", 1), (\"store\", \"x\")]\n    assert possible_locked({\"A\": inc, \"B\": inc}, {\"x\": 0}, \"x\") == [1, 2]\n    assert possible_locked({\"A\": protect(inc), \"B\": protect(inc)}, {\"x\": 0}, \"x\") == [2]\n    assert possible_locked({\"A\": protect(inc), \"B\": protect(inc), \"C\": protect(inc)}, {\"x\": 0}, \"x\") == [3]\n    assert possible_locked({\"A\": protect(inc), \"B\": inc}, {\"x\": 0}, \"x\") == [1, 2]\n    assert protect([(\"add\", 1)], \"M\") == [(\"acquire\", \"M\"), (\"add\", 1), (\"release\", \"M\")]")

### Exercise: Locks in a consistent order

Write `transfer(accounts, locks, src, dst, amount)`: move `amount` from `accounts[src]` to `accounts[dst]` while holding **both** accounts' locks (from the dict `locks`), always acquiring them in **sorted order of account name** with `with` statements, whichever direction the money goes. Raise `ValueError("insufficient funds")` if the source can't cover it (leaving balances unchanged), and `ValueError("same account")` if `src == dst`. The tests pass in locks that record the order they're taken in.

In [ ]:
def transfer(accounts, locks, src, dst, amount):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_transfer():\n    \"lock ordering\"\n    import threading\n    taken = []\n    class Recorded:\n        def __init__(self, name):\n            self.name, self.lock = name, threading.Lock()\n        def __enter__(self):\n            self.lock.acquire()\n            taken.append(self.name)\n            return self\n        def __exit__(self, *exc):\n            self.lock.release()\n            return False\n    accounts = {\"ada\": 100, \"ben\": 50}\n    locks = {name: Recorded(name) for name in accounts}\n    transfer(accounts, locks, \"ben\", \"ada\", 20)\n    transfer(accounts, locks, \"ada\", \"ben\", 70)\n    assert accounts == {\"ada\": 50, \"ben\": 100} and taken == [\"ada\", \"ben\", \"ada\", \"ben\"]\n    try:\n        transfer(accounts, locks, \"ada\", \"ben\", 1000)\n        assert False\n    except ValueError as e:\n        assert str(e) == \"insufficient funds\"\n    assert accounts == {\"ada\": 50, \"ben\": 100} and not locks[\"ada\"].lock.locked()\n    try:\n        transfer(accounts, locks, \"ada\", \"ada\", 1)\n        assert False\n    except ValueError as e:\n        assert str(e) == \"same account\"")

### Exercise: Finding a deadlock

Write `waits_for(holds, wants)`: the wait-for graph as a dict from each waiting thread to the thread holding the lock it wants (threads waiting for a free lock aren't waiting for anyone, so leave them out). `holds` maps lock to holding thread; `wants` maps thread to the lock it's waiting for. Then `deadlocked(holds, wants)`: the sorted list of threads that are part of a **cycle** in that graph (empty if there's no deadlock).

In [ ]:
def waits_for(holds, wants):
    ___

def deadlocked(holds, wants):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_deadlock():\n    \"wait-for graphs\"\n    holds = {\"A\": \"t1\", \"B\": \"t2\", \"C\": \"t3\"}\n    wants = {\"t1\": \"B\", \"t2\": \"C\", \"t3\": \"A\", \"t4\": \"A\"}\n    assert waits_for(holds, wants) == {\"t1\": \"t2\", \"t2\": \"t3\", \"t3\": \"t1\", \"t4\": \"t1\"}\n    assert deadlocked(holds, wants) == [\"t1\", \"t2\", \"t3\"]\n    assert deadlocked({\"A\": \"t1\"}, {\"t2\": \"A\"}) == [] and waits_for({}, {\"t1\": \"Z\"}) == {}\n    assert deadlocked({\"A\": \"t1\", \"B\": \"t2\"}, {\"t1\": \"B\", \"t2\": \"A\", \"t3\": \"B\"}) == [\"t1\", \"t2\"]")

### Bug hunt: A lock that waits for itself

`log_all` holds the lock and then calls `log`, which tries to take the **same** plain `Lock` again; a `Lock` isn't re-entrant, so the second acquire can never succeed (here it gives up after a short timeout) and this prints `stuck`. Use an `RLock`, which the owning thread may acquire again, so it prints `ok`.

In [ ]:
import threading

lock = threading.Lock()

def log(lines, message):
    if not lock.acquire(timeout=0.2):
        return False
    try:
        lines.append(message)
        return True
    finally:
        lock.release()

def log_all(lines, messages):
    with lock:
        return all(log(lines, m) for m in messages)

print("ok" if log_all([], ["a", "b"]) else "stuck")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a re-entrant lock\"\n    expect_output(\"ok\")")

### Bug hunt: A lock never released

`update` acquires the lock by hand, and when the update raises, `release()` is never reached, so the lock stays held for ever and the next update can't get it: this prints `locked out`. Use `with lock:` so the lock is released even when an error happens, and this prints `updated`.

In [ ]:
import threading

lock = threading.Lock()
stock = {"tea": 5}

def update(item, change):
    lock.acquire()
    stock[item] += change
    lock.release()

try:
    update("jam", 1)
except KeyError:
    pass
print("updated" if lock.acquire(timeout=0.2) else "locked out")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"with releases\"\n    expect_output(\"updated\")")

### Bug hunt: Two orders, one deadlock

Each transfer locks the **source** account first, so a transfer from x to y and one from y to x take the locks in opposite orders, which can deadlock: this prints `deadlock possible`. Lock the accounts in sorted order instead, whatever the direction, so it prints `safe`.

In [ ]:
def lock_order(src, dst):
    return (src, dst)

first = lock_order("x", "y")
second = lock_order("y", "x")
print("safe" if first == second else "deadlock possible")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a global order\"\n    expect_output(\"safe\")")

### Puzzle: A safe withdrawal

Arrange the lines of a method that withdraws money only if there's enough, with the check and the change inside one lock. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # possible final x when two threads each do load x, add 1, store x, starting at 0, as a sorted list
q = ___   # what lock.acquire(blocking=False) returns when another thread holds it
r = ___   # the lock a thread can acquire again while holding it: "Lock" or "RLock"
s = ___   # how many interleavings two threads of 2 steps each have
t = ___   # what a deadlock is in the wait-for graph: "a cycle" or "a long path"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [1, 2] and q is False and r == \"RLock\", \"p, q and r\"\n    assert s == 6 and t == \"a cycle\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a bank under a random scheduler

Test concurrency code the way race detectors do: run it under many **random** schedules. Write:

- `SimLock`, a simulated lock with a `holder` (or `None`);
- `run_random(threads, seed)`: threads are a dict of generators that yield `("acquire", lock)`, `("release", lock)` or `None` (a plain step). At each step, pick a **random** runnable thread with `random.Random(seed)` (choosing from runnable names in the dict's order); a thread is runnable unless its pending request is an acquire of a lock another thread holds. A request takes effect when that thread is chosen. Run until all finish, or raise `RuntimeError("deadlock")` if unfinished threads exist but none is runnable;
- `transfer_thread(accounts, src, dst, amount, locks=None, ordered=True)`: a generator thread that moves money from `src` to `dst` by reading both balances, yielding (a plain step: a moment another thread may run), and then writing both. With `locks` (a dict of `SimLock`s), it acquires both accounts' locks first (sorted by name if `ordered`, otherwise source first) and releases them at the end.

The tests run many seeds: without locks, schedules lose or create money; unordered locks deadlock under some schedule; ordered locks always conserve money and never deadlock.

In [ ]:
import random

class SimLock:
    def __init__(self):
        ___

def run_random(threads, seed):
    ___

def transfer_thread(accounts, src, dst, amount, locks=None, ordered=True):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_bank():\n    \"random schedules\"\n    def trial(seed, locked, ordered):\n        accounts = {\"x\": 100, \"y\": 100, \"z\": 100}\n        locks = {n: SimLock() for n in accounts} if locked else None\n        jobs = [(\"x\", \"y\", 10), (\"y\", \"x\", 20), (\"y\", \"z\", 5), (\"z\", \"x\", 15)]\n        threads = {f\"t{i}\": transfer_thread(accounts, s, d, a, locks, ordered) for i, (s, d, a) in enumerate(jobs)}\n        try:\n            run_random(threads, seed)\n        except RuntimeError as e:\n            return str(e)\n        return sum(accounts.values())\n    unlocked = {trial(s, False, True) for s in range(200)}\n    assert any(v != 300 for v in unlocked)\n    unordered = {trial(s, True, False) for s in range(200)}\n    assert \"deadlock\" in unordered\n    ordered = {trial(s, True, True) for s in range(200)}\n    assert ordered == {300}\n\ndef test_scheduler():\n    \"the scheduler itself\"\n    log = []\n    def worker(name, lock):\n        yield (\"acquire\", lock)\n        log.append(name + \" in\")\n        yield None\n        log.append(name + \" out\")\n        yield (\"release\", lock)\n    lock = SimLock()\n    run_random({\"a\": worker(\"a\", lock), \"b\": worker(\"b\", lock)}, seed=1)\n    assert len(log) == 4 and log[0][0] == log[1][0] and log[2][0] == log[3][0] and lock.holder is None")

## Warm-up

Two threads each add one to `x` with a load and a store. There are six possible orders, and in four of them one update is lost, so `x` ends up 1 instead of 2. Nothing about the code is wrong line by line; the bug is in the interleaving.

## Race conditions

A **race condition** is a bug where the outcome depends on the relative timing of threads. The classic form is **read-modify-write**: `counter += 1`, `balance -= amount`, `items[key].append(x)` after a lookup. Each is several steps, and another thread can run between them.

Key words:

- An operation is **atomic** if no other thread can observe it half done.
- A **critical section** is a piece of code that must not run in two threads at once (because it reads and writes shared data).
- **Shared mutable state** is the root cause. No shared state (each thread uses its own data and sends results through a queue) means no races.

Races are hard to find by testing, because the bad interleavings may be rare. Ways to reason about them: imagine the worst moment for a switch (between every pair of steps), **enumerate** interleavings for small cases (two threads with `m` and `n` steps have `(m + n)! / (m! n!)` orders), and use tools: the free-threaded build and race detectors make races show up more often.

## Locks

A **lock** (`threading.Lock`, a "mutex") is held by at most one thread at a time. `acquire()` takes it, waiting while another thread holds it; `release()` gives it back. Wrapping a critical section in a lock makes it effectively atomic with respect to other code that uses the **same** lock.

Always use the `with` statement (Week 61), which releases the lock even if the block raises:

```
balance_lock = threading.Lock()

def withdraw(amount):
    with balance_lock:                   # one thread at a time in here
        if balance >= amount:
            balance -= amount
            return True
        return False
```

- `acquire(blocking=False)` returns `False` immediately if the lock is taken; `acquire(timeout=1.0)` waits at most a second; both return `True` on success.
- A plain `Lock` isn't **re-entrant**: if a thread holding it calls `acquire()` again, it waits for itself for ever. **`threading.RLock`** (re-entrant lock) counts acquisitions by the owning thread and can be acquired again by it; it must be released as many times. Use it when a method holding the lock calls another method that takes the same lock.
- Locks protect data only if **every** access goes through them. One unlocked read or write, anywhere, and the race is back.
- Keep critical sections **short**: while one thread holds the lock, others that need it wait, so long sections remove the benefit of threads.

## Check-then-act and other patterns

Races aren't only about counters. Watch for:

- **Check-then-act**: `if key not in cache: cache[key] = load(key)` (two threads both see it missing and both load); `if os.path.exists(path): open(path)` (the file can vanish in between); `if balance >= amount: balance -= amount`. Put the check **and** the act inside one lock, or use an operation that does both at once (`dict.setdefault`, opening a file with mode `"x"`, a database transaction).
- **Compound operations on several objects**: moving an item from one list to another must hold a lock covering both.
- **Lazy initialisation**: creating a shared object on first use (`if self._conn is None: self._conn = connect()`) needs a lock, or do it before starting threads.
- **Iterating while another thread modifies** a list or dict: copy it under the lock first.

And the simplest fix of all: **don't share**. Give each thread its own data, use `queue.Queue` to hand work and results between threads, or make shared data **immutable** (Week 62): immutable objects can be read by any number of threads safely.

## Deadlock

Locks introduce a new failure: **deadlock**, where threads wait for each other for ever. The classic case: thread 1 holds lock A and waits for lock B, while thread 2 holds B and waits for A. Neither can proceed. Four conditions together make deadlock possible (Coffman's conditions): locks held exclusively, held while waiting for more, not taken away by force, and a **cycle** of threads each waiting for the next.

Prevention, most useful first:

- **A global lock order**: whenever a thread needs several locks, it takes them in one agreed order (by name, by id, by account number). A cycle can then never form. A bank transfer between accounts `x` and `y` locks `min(x, y)` first, whichever way the money moves.
- **Hold fewer locks**: one lock for related data is often simpler than several fine-grained ones.
- **Timeouts**: `acquire(timeout=...)` and give up (releasing what you hold) if it fails, then retry.
- **Never call unknown code while holding a lock** (a callback could try to take another lock).

**Detecting** deadlock: build a **wait-for graph** with an edge from each waiting thread to the thread holding the lock it wants. A deadlock is a **cycle** in that graph (Week 50's cycle detection).

## More primitives

`threading` has a few more tools, all usable with `with`:

- **`Semaphore(n)`**: a lock that up to `n` threads can hold at once. Use it to **limit concurrency**: at most 5 simultaneous downloads, at most 10 database connections. `BoundedSemaphore` also catches releasing too many times.
- **`Condition`**: a lock plus the ability to **wait for a condition** to become true: `with cond: while not ready(): cond.wait()`, and another thread changes the state and calls `cond.notify()` or `notify_all()`. It's what `queue.Queue` is built on; prefer a queue when one fits.
- **`Event`** (Tuesday): a one-off flag.
- **`Barrier(n)`**: makes `n` threads wait until all of them have arrived, then releases them together.

Rules of thumb: prefer **queues** for passing data, **one lock** around a small piece of shared state when you can't avoid sharing, **semaphores** for limits, and don't invent your own synchronisation from flags and `sleep`.

## Practice

## Quiz

## Daily challenge

## Before you go

On your own computer, run two threads that each do `counter += 1` a million times on a shared global, with and without a `Lock`, and compare the final counts. (With the GIL, you may need several runs, or a free-threaded build, to see a lost update.) Tomorrow: `concurrent.futures`, the high-level way to run work in threads and processes.